# scVI / scANVI fine-tuning — Mouse Cardiac snRNA-seq

## Datasets

| Role | File | Samples | Cell types |
|------|------|---------|------------|
| **Reference (train)** | `reference_adult_aged.h5ad` | 9w adult, 78w aged (ASE dataset) | FB, ventCM, EC, SMC, Macro, atrialCM, LEC, Epicard, Adipo, Glial |
| **Query (predict)** | `query_OCM.h5ad` | 9w, 78w, TAC (28d), Sham (28d) | unlabelled |

## Strategy

| Step | Method | Purpose |
|------|--------|---------|
| 1 | **scVI** on reference | Learns latent space; corrects adult vs aged batch |
| 2 | **scANVI** on reference | Adds semi-supervised cell-type classifier |
| 3 | **scArches surgery** on query | Grafts new per-sample encoder embeddings; decoder frozen |
| 4 | **Fine-tune** query model | ~200 epochs (much less than from scratch) |
| 5 | **Predict** cell types | Soft probabilities + hard labels for every OCM cell |

**Prerequisites:** Run `OCM_heart/export_seurat_to_h5ad.R` locally, then upload both `.h5ad` files to Google Drive.

> **Runtime:** GPU — Runtime → Change runtime type → T4 GPU

## 0. Install dependencies

In [ ]:
# Only needs to run once per Colab session
# scib is optional; skip it unless you want benchmarking metrics
!pip install -q scvi-tools scarches
# Restart runtime after install: Runtime → Restart session

In [3]:
!pip install scvi-tools

!pip install scib
!pip install scarches

  Using cached scib-1.1.7-1-py3-none-any.whl.metadata (9.8 kB)
  Using cached leidenalg-0.12.0-cp38-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached igraph-1.0.0-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (4.4 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached texttable-1.7.0-py2.py3-none-any.whl.metadata (9.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 66.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 105.8 MB/s eta 0:00:00
  Using cached scArches-0.6.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached scHPL-1.0.5-py3-none-any.whl.metadata (3.5 kB)
  Using cached muon-0.1.7-py3-none-any.whl.metadata (7.2 kB)
INFO: pip is looking at multiple versions of schpl to determine which version is compatible with other requirements. This could take a while.
  Using cached scHPL-1.0.4-py3-non

## 1. Mount Google Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit this to the folder where you uploaded the .h5ad files ──────
DATA_DIR = '/content/drive/MyDrive/cardiac_scVI'
OUT_DIR  = '/content/drive/MyDrive/cardiac_scVI/results'
os.makedirs(OUT_DIR, exist_ok=True)

REF_H5AD   = os.path.join(DATA_DIR, 'reference_adult_aged.h5ad')
QUERY_H5AD = os.path.join(DATA_DIR, 'query_OCM.h5ad')

# Column in reference obs holding cell-type annotation
CELLTYPE_KEY = 'celltype'         # values: FB, ventCM, EC, SMC, Macro, atrialCM, LEC, Epicard, Adipo, Glial

# Batch correction columns
REF_BATCH_KEY   = 'age'           # reference: 'adult' | 'aged'
QUERY_BATCH_KEY = 'sample'        # query:     '9w' | '78w' | 'TAC' | 'Sham'

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import scarches as sca
import torch

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, color_map='viridis')

print(f'scvi-tools : {scvi.__version__}')
print(f'scarches   : {sca.__version__}')
print(f'GPU        : {torch.cuda.is_available()}')

## 3. Load data

In [ ]:
ref   = sc.read_h5ad(REF_H5AD)
query = sc.read_h5ad(QUERY_H5AD)

print('Reference:', ref)
print()
print('Query:    ', query)

# Confirm annotation column and cell types
assert CELLTYPE_KEY in ref.obs.columns, (
    f"'{CELLTYPE_KEY}' not in reference obs. Available: {list(ref.obs.columns)}"
)
print(f"\nReference cell types ({CELLTYPE_KEY}):")
print(ref.obs[CELLTYPE_KEY].value_counts())
print(f"\nReference batches ({REF_BATCH_KEY}):")
print(ref.obs[REF_BATCH_KEY].value_counts())
print(f"\nQuery batches ({QUERY_BATCH_KEY}):")
print(query.obs[QUERY_BATCH_KEY].value_counts())

## 4. Quality control (light — CellBender already cleaned the data)

In [ ]:
import scipy.sparse as sp

def activate_raw_counts(adata, name):
    """Place raw (integer) counts in .X — required by scVI."""
    # SeuratDisk typically stores raw counts in layers['counts']
    if 'counts' in adata.layers:
        adata.X = adata.layers['counts'].copy()
        print(f"{name}: using layers['counts']")
    elif adata.raw is not None:
        adata.X = adata.raw.X.copy()
        print(f"{name}: using .raw.X")
    else:
        print(f"{name}: using .X as-is — verify these are raw counts")

    X = adata.X
    if sp.issparse(X):
        X.data = np.round(X.data).astype(np.float32)
    else:
        adata.X = np.round(X).astype(np.float32)

    total = float(adata.X.sum())
    print(f"  {adata.n_obs} cells x {adata.n_vars} genes | total counts = {total:,.0f}")

activate_raw_counts(ref,   'reference')
activate_raw_counts(query, 'query')

# Basic per-cell stats
sc.pp.calculate_qc_metrics(ref,   inplace=True)
sc.pp.calculate_qc_metrics(query, inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, adata, title in zip(axes, [ref, query],
                             ['Reference (adult + aged)', 'Query (OCM: 9w / 78w / TAC / Sham)']):
    ax.hist(adata.obs['total_counts'], bins=50, log=True)
    ax.set_xlabel('Total counts'); ax.set_ylabel('Cells'); ax.set_title(title)
plt.tight_layout(); plt.show()

## 5. Highly variable gene selection on the joint feature space

In [ ]:
N_HVG = 3000

# Concatenate just to compute HVGs on the shared gene universe
query.obs[CELLTYPE_KEY] = 'Unknown'  # placeholder for unlabelled query cells
combined = sc.concat(
    {'ref': ref, 'query': query},
    label='_source', keys=['ref', 'query'],
    join='inner'   # keep only genes present in BOTH datasets
)
print(f'Shared gene space: {combined.n_vars} genes')

# Normalise a copy for HVG scoring only (scVI always uses raw counts)
tmp = combined.copy()
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
sc.pp.highly_variable_genes(
    tmp,
    n_top_genes=N_HVG,
    flavor='seurat_v3',
    batch_key='_source',
    subset=False
)
hvg = tmp.var['highly_variable']
print(f'HVGs selected: {hvg.sum()}')
del tmp

# Subset both to the same HVG set
ref_hvg   = ref[:, hvg].copy()
query_hvg = query[:, hvg].copy()

# Add a unified 'batch' column used by scVI setup
# Reference: adult / aged  →  Query: 9w / 78w / TAC / Sham
ref_hvg.obs['batch']   = ref_hvg.obs[REF_BATCH_KEY].astype(str)
query_hvg.obs['batch'] = query_hvg.obs[QUERY_BATCH_KEY].astype(str)

## 6. Train scVI on reference (adult + aged)

Learns a batch-corrected (adult vs aged) latent space using raw counts and a negative-binomial likelihood — well-suited for sparse snRNA-seq data.

In [ ]:
scvi.model.SCVI.setup_anndata(
    ref_hvg,
    batch_key='batch',   # corrects adult vs aged
)

model_ref = scvi.model.SCVI(
    ref_hvg,
    n_layers=2,
    n_latent=30,
    gene_likelihood='nb',
)
print(model_ref)

model_ref.train(
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=20,
    batch_size=256,
    plan_kwargs={'lr': 1e-3},
)

plt.figure(figsize=(7, 3))
plt.plot(model_ref.history['elbo_train'],      label='train')
plt.plot(model_ref.history['elbo_validation'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('ELBO'); plt.legend()
plt.title('scVI reference — training curve'); plt.tight_layout(); plt.show()

model_ref.save(os.path.join(OUT_DIR, 'scVI_reference'), overwrite=True)
print('scVI reference model saved.')

## 7. Extend to scANVI — semi-supervised cell-type classifier

scANVI wraps scVI and adds a classifier trained on the 10 annotated cell types from the reference.
The `'Unknown'` category is the placeholder for future unlabelled data.

In [ ]:
model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_ref,
    unlabeled_category='Unknown',
    labels_key=CELLTYPE_KEY,
)
print(model_scanvi)

model_scanvi.train(
    max_epochs=200,
    n_samples_per_label=20,
    early_stopping=True,
    early_stopping_patience=15,
    batch_size=256,
)

# Reference latent space
ref_hvg.obsm['X_scANVI'] = model_scanvi.get_latent_representation()

sc.pp.neighbors(ref_hvg, use_rep='X_scANVI', n_neighbors=30)
sc.tl.umap(ref_hvg)

# Matched to the cluster_colors palette in adult_aged_snRNAseq.R
cell_type_palette = {
    'Glial':    '#EC68A3',
    'FB':       '#459AD5',
    'LEC':      '#11B6EB',
    'Adipo':    '#9188C0',
    'SMC':      '#2EB7BE',
    'Epicard':  '#AD7AB3',
    'Macro':    '#36B28F',
    'atrialCM': '#59B031',
    'EC':       '#9AA921',
    'ventCM':   '#EE766F',
    'Unknown':  '#AAAAAA',
}

sc.pl.umap(ref_hvg,
           color=[CELLTYPE_KEY, 'batch'],
           palette=cell_type_palette,
           title=['Reference cell types', 'Reference batch (age)'])

model_scanvi.save(os.path.join(OUT_DIR, 'scANVI_reference'), overwrite=True)
print('scANVI reference model saved.')

## 8. scArches surgery — fine-tune on OCM query data

**Surgery** grafts new encoder embeddings for the four OCM sample batches (9w, 78w, TAC, Sham) while **keeping the reference decoder frozen**. This preserves the latent geometry learned from the reference and requires far fewer epochs.

In [ ]:
# Register query data with the same setup as the reference
# 'Unknown' is the unlabelled placeholder
query_hvg.obs[CELLTYPE_KEY] = 'Unknown'

model_query = sca.models.SCANVI.load_query_data(
    query_hvg,
    reference_model=model_scanvi,   # the fine-tuned scANVI reference
    labeled_indices=[],             # all query cells are unlabelled
    freeze_decoder_first_layer=True,
    freeze_batchnorm_encoder=True,
    freeze_batchnorm_decoder=False,
    freeze_expression=True,         # freeze reference encoder — only learn new batch embeddings
)

print(model_query)

# Fine-tune — far fewer epochs needed compared to training from scratch
model_query.train(
    max_epochs=200,
    plan_kwargs={'weight_decay': 0.0, 'lr': 1e-3},
    early_stopping=True,
    early_stopping_patience=15,
    batch_size=256,
)

# Training curve
plt.plot(model_query.history['elbo_train'], label='ELBO train')
plt.xlabel('Epoch'); plt.ylabel('ELBO'); plt.legend(); plt.title('scArches query fine-tuning')
plt.show()

model_query.save(os.path.join(OUT_DIR, 'scANVI_query_model'), overwrite=True)
print('Query model saved.')

## 9. Predict cell types on OCM query cells

In [ ]:
soft_pred = model_query.predict(soft=True)
query_hvg.obs['predicted_celltype'] = model_query.predict(soft=False).values

probs = soft_pred.values
entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
query_hvg.obs['prediction_uncertainty'] = entropy

print('Predicted cell type counts (all OCM samples):')
print(query_hvg.obs['predicted_celltype'].value_counts())
print('\nPer sample breakdown:')
print(pd.crosstab(query_hvg.obs['predicted_celltype'], query_hvg.obs[QUERY_BATCH_KEY]))

## 10. Joint UMAP — reference + query in shared latent space

In [ ]:
lat_ref   = model_scanvi.get_latent_representation(ref_hvg)
lat_query = model_query.get_latent_representation(query_hvg)

combined_hvg = sc.concat(
    {'reference': ref_hvg, 'query': query_hvg},
    label='dataset', keys=['reference', 'query'],
    join='inner'
)
combined_hvg.obsm['X_scANVI'] = np.vstack([lat_ref, lat_query])
combined_hvg.obs['celltype_label'] = pd.concat([
    ref_hvg.obs[CELLTYPE_KEY],
    query_hvg.obs['predicted_celltype']
])

sc.pp.neighbors(combined_hvg, use_rep='X_scANVI', n_neighbors=30)
sc.tl.umap(combined_hvg, min_dist=0.3)

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sc.pl.umap(combined_hvg, color='dataset',        ax=axes[0], show=False, title='Dataset')
sc.pl.umap(combined_hvg, color='celltype_label', ax=axes[1], show=False,
           palette=cell_type_palette,
           title='Cell type (ref labels / query predictions)')
sc.pl.umap(combined_hvg, color='batch',          ax=axes[2], show=False, title='Batch / sample')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'joint_UMAP.pdf'), bbox_inches='tight')
plt.show()

## 11. Cell-type composition per sample

In [ ]:
def composition_bar(adata, group_col, color_col, ax, title):
    comp = (adata.obs
            .groupby([group_col, color_col], observed=True)
            .size()
            .unstack(fill_value=0)
            .apply(lambda x: x / x.sum(), axis=1))
    colors = [cell_type_palette.get(c, '#AAAAAA') for c in comp.columns]
    comp.plot(kind='bar', stacked=True, ax=ax, color=colors, legend=False, width=0.7)
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('Proportion')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
composition_bar(ref_hvg,   'batch',           CELLTYPE_KEY,        axes[0], 'Reference — annotated (adult / aged)')
composition_bar(query_hvg, QUERY_BATCH_KEY,   'predicted_celltype', axes[1], 'OCM query — predicted (9w / 78w / TAC / Sham)')

handles = [plt.Rectangle((0,0),1,1, color=cell_type_palette[ct])
           for ct in cell_type_palette if ct != 'Unknown']
fig.legend(handles, [ct for ct in cell_type_palette if ct != 'Unknown'],
           loc='center right', bbox_to_anchor=(1.15, 0.5), title='Cell type')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'celltype_composition.pdf'), bbox_inches='tight')
plt.show()

## 12. Prediction uncertainty filter

In [ ]:
# Inspect uncertainty distribution
plt.figure(figsize=(6, 3))
plt.hist(query_hvg.obs['prediction_uncertainty'], bins=50, color='steelblue')
plt.axvline(np.percentile(query_hvg.obs['prediction_uncertainty'], 90),
            color='red', linestyle='--', label='90th percentile')
plt.xlabel('Entropy (prediction uncertainty)')
plt.ylabel('Cells')
plt.title('Query prediction uncertainty')
plt.legend(); plt.tight_layout(); plt.show()

# High-confidence predictions (bottom 90% uncertainty)
thresh = np.percentile(query_hvg.obs['prediction_uncertainty'], 90)
confident = query_hvg[query_hvg.obs['prediction_uncertainty'] < thresh]
print(f"High-confidence predictions: {confident.n_obs} / {query_hvg.n_obs} cells ({100*confident.n_obs/query_hvg.n_obs:.1f}%)")
print(confident.obs['predicted_celltype'].value_counts())

## 13. Save results

In [ ]:
query_hvg.write_h5ad(os.path.join(OUT_DIR, 'query_OCM_predicted.h5ad'))

pred_df = query_hvg.obs[['predicted_celltype', 'prediction_uncertainty', QUERY_BATCH_KEY]].copy()
pred_df.index.name = 'cell_barcode'
pred_df.to_csv(os.path.join(OUT_DIR, 'query_OCM_predictions.csv'))

print('Files written to', OUT_DIR)
print('  query_OCM_predicted.h5ad')
print('  query_OCM_predictions.csv')

## 14. Re-import predictions into Seurat (run locally in R)

```r
library(Seurat)
library(readr)

pred <- read_csv('results/query_OCM_predictions.csv') |>
  tibble::column_to_rownames('cell_barcode')

heart <- readRDS('OCM_heart/heart_seurat_object_SCT.rds')
heart$scANVI_celltype    <- NA_character_
heart$scANVI_uncertainty <- NA_real_

common <- intersect(rownames(pred), colnames(heart))
heart$scANVI_celltype[match(common, colnames(heart))]    <- pred[common, 'predicted_celltype']
heart$scANVI_uncertainty[match(common, colnames(heart))] <- pred[common, 'prediction_uncertainty']

# Visualise — split by sample to compare 9w / 78w / TAC / Sham
DimPlot(heart, group.by = 'scANVI_celltype', split.by = 'sample', label = TRUE)
```